# Jour 2 : adapter un petit modèle de langage au swahili avec LoRA

**Deep Learning IndabaX RDC 2026 · Atelier PyTorch**
Université Nouveaux Horizons (UNH), Lubumbashi · 27 novembre 2026 · **session de 30 minutes**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganji759/IndabaX-DRC-2026-PyTorch-Workshop/blob/main/notebooks/Jour2_Finetuning_LoRA.ipynb)

Hier, nous avons appris les bases de PyTorch ([notebook du Jour 1](https://colab.research.google.com/github/ganji759/IndabaX-DRC-2026-PyTorch-Workshop/blob/main/notebooks/Jour1_Introduction_PyTorch.ipynb)).
Nous avons fini avec une idée : **un vrai modèle de langage s'entraîne avec la même boucle**.
Aujourd'hui, nous le faisons. Nous prenons un vrai modèle pré-entraîné, et nous l'adaptons à des instructions en swahili.

### Objectifs

À la fin de ce notebook, vous savez :

1. charger un modèle de langage pré-entraîné et lui poser une question ;
2. préparer des conversations pour l'entraînement ;
3. expliquer **LoRA** : pourquoi on entraîne seulement quelques petits poids ;
4. fine-tuner le modèle avec **la boucle d'entraînement d'hier** ;
5. comparer les réponses **avant** et **après** ;
6. sauvegarder le résultat et comprendre comment le déployer sur un téléphone avec **ExecuTorch**.

<p align="center"><img src="https://raw.githubusercontent.com/ganji759/IndabaX-DRC-2026-PyTorch-Workshop/main/images/11_jour2_parcours.png" alt="Les 7 étapes du Jour 2" width="900"></p>

| Partie | Sujet | Durée |
|---|---|---|
| 0 | Mise en place | 3 min |
| 1 | Le modèle pré-entraîné et les réponses « avant » | 5 min |
| 2 | Les données en swahili | 4 min |
| 3 | LoRA | 5 min |
| 4 | L'entraînement | 7 min |
| 5 | Les réponses « après » | 3 min |
| 6 | Sauvegarder et déployer | 3 min |

> **Conseil pour gagner du temps :** activez le GPU (ci-dessous), puis lancez **Exécution → Tout exécuter** (*Runtime → Run all*) dès le début.
> Le notebook tourne pendant que nous expliquons chaque partie. L'entraînement prend quelques minutes sur le GPU T4.

> **Soyons honnêtes :** le modèle est petit (0,5 milliard de paramètres) et l'entraînement est court.
> Les réponses resteront imparfaites. Le but est d'apprendre **le processus complet**, pas d'obtenir un modèle parfait.

## 0. Mise en place

1. Menu **Exécution → Modifier le type d'exécution** → **GPU T4** → **Enregistrer**.
2. Exécutez les cellules dans l'ordre (**Maj + Entrée**), ou tout d'un coup (**Exécution → Tout exécuter**).

Colab a déjà PyTorch, `transformers` (les modèles de Hugging Face) et `datasets`. Nous ajoutons `peft`, la bibliothèque qui contient LoRA.

In [ ]:
# Install the LoRA library (PEFT). PyTorch, transformers and datasets are already in Colab.
!pip install -q peft

In [ ]:
import time

import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device: {device}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU: the notebook still runs, with a much smaller training set.")

_ = torch.manual_seed(42)

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"         # 0.5 billion parameters, Apache-2.0
DATASET_ID = "regnant-io/swahili-instruction-22k"  # 22,500 Swahili conversations, Apache-2.0

# Settings that keep the session short
N_TRAIN = 1000 if device == "cuda" else 40   # number of conversations used for training
BATCH_SIZE = 8 if device == "cuda" else 2
MAX_LEN = 256                                # max tokens per conversation
LEARNING_RATE = 2e-4
EPOCHS = 1

## 1. Le modèle pré-entraîné

Nous utilisons **Qwen2.5-0.5B-Instruct** : un petit modèle de langage déjà entraîné sur des milliards de mots, surtout en anglais et en chinois.
Il sait déjà converser. Il connaît un peu le swahili, mais pas très bien.

Hier, nous avons construit un modèle avec `nn.Module`. Ici, c'est pareil : `AutoModelForCausalLM` renvoie un `nn.Module` PyTorch, simplement beaucoup plus grand.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params / 1e6:.0f} million")
print("It is a PyTorch nn.Module:", isinstance(model, torch.nn.Module))

Pour discuter avec le modèle, le **tokenizer** met la conversation dans un format précis, avec des **tokens spéciaux** (`<|im_start|>`, `<|im_end|>`).
C'est le **chat template** du modèle.

In [ ]:
messages = [{"role": "user", "content": "Habari ya asubuhi!"}]
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

In [ ]:
def ask(model, question, max_new_tokens=100):
    """Ask the model one question and return its answer (greedy decoding)."""
    messages = [{"role": "user", "content": question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    model.eval()
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                do_sample=False, repetition_penalty=1.1)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

Posons trois questions en swahili, **avant** l'entraînement. Nous gardons les réponses pour les comparer à la fin.

| Swahili | Français |
|---|---|
| Eleza kwa ufupi akili bandia ni nini. | Explique brièvement ce qu'est l'intelligence artificielle. |
| Nipe vidokezo vitatu vya kuwa na afya njema. | Donne-moi trois conseils pour être en bonne santé. |
| Taja nchi tano za Afrika. | Cite cinq pays d'Afrique. |

In [ ]:
QUESTIONS = [
    "Eleza kwa ufupi akili bandia ni nini.",
    "Nipe vidokezo vitatu vya kuwa na afya njema.",
    "Taja nchi tano za Afrika.",
]

answers_before = {}
for q in QUESTIONS:
    answers_before[q] = ask(model, q)
    print(f"Q: {q}\nA (BEFORE): {answers_before[q]}\n" + "-" * 70)

## 2. Les données en swahili

Le jeu de données [`regnant-io/swahili-instruction-22k`](https://huggingface.co/datasets/regnant-io/swahili-instruction-22k)
contient 22 500 conversations en swahili : une instruction (`user`) et une réponse (`assistant`).

> Ces données ont été **traduites automatiquement** de l'anglais (modèle NLLB-200). Elles contiennent donc des erreurs.
> Pour un vrai projet, des données écrites ou vérifiées par des locuteurs natifs sont bien meilleures.

Certaines conversations ont plusieurs tours. Nous gardons seulement les conversations à **un seul tour** (une question, une réponse).
Pour tenir en 30 minutes, nous en prenons **1 000** au hasard.

In [ ]:
dataset = load_dataset(DATASET_ID, split="train")
print(dataset)

for row in dataset.select(range(2)):
    for message in row["messages"]:
        print(f"[{message['role']}] {message['content']}")
    print("-" * 70)

# Keep only single-turn conversations: one question, one answer
single = dataset.filter(lambda e: len(e["messages"]) == 2)
print(f"Single-turn conversations: {len(single)}")

small = single.shuffle(seed=42).select(range(N_TRAIN))
print(f"Training on {len(small)} conversations")

### Préparer chaque conversation

<p align="center"><img src="https://raw.githubusercontent.com/ganji759/IndabaX-DRC-2026-PyTorch-Workshop/main/images/13_format_donnees.png" alt="Préparer une conversation : chat template, tokens et labels" width="900"></p>

1. On met la conversation dans le **chat template**, puis on la transforme en **tokens**.
2. On crée les **labels** (les bonnes réponses que le modèle doit prédire).
   Pour les tokens de la **question**, on met `-100` : la fonction de perte les **ignore**.
   Le modèle apprend donc seulement à écrire la **réponse**.

> Le chat template de Qwen ajoute aussi un message `system` par défaut au début (le schéma ne le montre pas). Il fait partie de la question : il est ignoré dans la perte.

In [ ]:
def format_example(example):
    messages = example["messages"]
    # The prompt = everything before the answer (with the "assistant" header)
    prompt_text = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"][:MAX_LEN]
    # -100 = "ignore this token in the loss" (same convention as nn.CrossEntropyLoss)
    labels = ([-100] * len(prompt_ids) + full_ids[len(prompt_ids):])[:len(full_ids)]
    return {"input_ids": full_ids, "labels": labels}

tokenized = small.map(format_example, remove_columns=small.column_names)
# Drop conversations whose answer was cut away by MAX_LEN
tokenized = tokenized.filter(lambda e: any(label != -100 for label in e["labels"]))
print(tokenized)

example = tokenized[0]
answer_ids = [t for t, label in zip(example["input_ids"], example["labels"]) if label != -100]
print("\nFULL TEXT  :", tokenizer.decode(example["input_ids"]))
print("\nLEARNED    :", tokenizer.decode(answer_ids))

Le `DataLoader` regroupe les conversations par **lots**, comme hier.
Les conversations n'ont pas la même longueur : la fonction `collate` les complète (*padding*) jusqu'à la plus longue du lot.

In [ ]:
def collate(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids = torch.full((len(batch), max_len), tokenizer.pad_token_id)
    labels = torch.full((len(batch), max_len), -100)
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)
    for i, x in enumerate(batch):
        n = len(x["input_ids"])
        input_ids[i, :n] = torch.tensor(x["input_ids"])
        labels[i, :n] = torch.tensor(x["labels"])
        attention_mask[i, :n] = 1          # 1 = real token, 0 = padding
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

train_dataloader = DataLoader(tokenized, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
batch = next(iter(train_dataloader))
print({k: tuple(v.shape) for k, v in batch.items()})
print(f"{len(train_dataloader)} batches per epoch")

## 3. LoRA : entraîner seulement de petits poids ajoutés

Entraîner les 494 millions de paramètres demande beaucoup de mémoire et de temps.
**LoRA** (*Low-Rank Adaptation*) a une idée simple :

- on **gèle** les poids `W` du modèle (ils ne changent pas) ;
- à côté de certaines couches `nn.Linear`, on ajoute deux **petites matrices** `A` et `B` ;
- on entraîne **seulement** `A` et `B`.

<p align="center"><img src="https://raw.githubusercontent.com/ganji759/IndabaX-DRC-2026-PyTorch-Workshop/main/images/12_lora.png" alt="LoRA : W est gelé, A et B sont appris" width="900"></p>

Au début, `B` vaut zéro : le modèle donne exactement les mêmes réponses qu'avant. L'entraînement modifie ensuite `A` et `B`.

> Vous vous rappelez `torch.no_grad()` et `requires_grad` d'hier ? LoRA met simplement `requires_grad=False` sur les poids d'origine.

Référence : [Hu et al., 2021, *LoRA: Low-Rank Adaptation of Large Language Models*](https://arxiv.org/abs/2106.09685).

In [ ]:
lora_config = LoraConfig(
    r=16,                 # rank: the size of the small matrices A and B
    lora_alpha=32,        # scaling factor of the update B @ A
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",   # attention layers
                    "gate_proj", "up_proj", "down_proj"],     # MLP layers
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Look inside one layer: the original weight is frozen, A and B are trainable
layer = model.base_model.model.model.layers[0].self_attn.q_proj
print("W (frozen) :", tuple(layer.base_layer.weight.shape), "requires_grad =", layer.base_layer.weight.requires_grad)
print("A (learned):", tuple(layer.lora_A["default"].weight.shape), "requires_grad =", layer.lora_A["default"].weight.requires_grad)
print("B (learned):", tuple(layer.lora_B["default"].weight.shape), "requires_grad =", layer.lora_B["default"].weight.requires_grad)
print("B is all zeros at the start:", bool((layer.lora_B["default"].weight == 0).all()))

## 4. L'entraînement : la même boucle qu'hier

<p align="center"><img src="https://raw.githubusercontent.com/ganji759/IndabaX-DRC-2026-PyTorch-Workshop/main/images/08_boucle_entrainement.png" alt="La boucle d'entraînement en 5 étapes" width="650"></p>

Comparez avec la boucle du Jour 1. Il y a seulement trois différences :

1. le modèle calcule la perte **lui-même** quand on lui donne les `labels` (`outputs.loss`, une `CrossEntropyLoss`) ;
2. l'optimiseur reçoit **seulement les paramètres LoRA** (ceux qui ont `requires_grad=True`) ;
3. sur GPU, nous utilisons la **précision mixte** (`torch.autocast`) : les calculs se font en `float16`, ce qui est plus rapide sur le T4.
   Voir le tutoriel officiel [Automatic Mixed Precision](https://docs.pytorch.org/tutorials/recipes/recipes/amp_recipe.html).

In [ ]:
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE)

use_amp = device == "cuda"                              # mixed precision only on the GPU
scaler = torch.amp.GradScaler(device, enabled=use_amp)  # avoids float16 underflow in gradients

losses = []
model.train()
start = time.perf_counter()
for epoch in range(EPOCHS):
    for step, batch in enumerate(train_dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}       # 1. take a batch

        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            outputs = model(**batch)                               # 2. prediction
            loss = outputs.loss                                    # 3. loss

        scaler.scale(loss).backward()                              # 4. gradients
        scaler.step(optimizer)                                     # 5. update
        scaler.update()
        optimizer.zero_grad()                                      #    reset

        losses.append(loss.item())
        if step % 10 == 0:
            print(f"step {step:4d}/{len(train_dataloader)} | loss {loss.item():.3f} | {time.perf_counter() - start:5.0f} s")

print(f"Done in {time.perf_counter() - start:.0f} s")

In [ ]:
window = 10
smooth = [sum(losses[max(0, i - window + 1):i + 1]) / len(losses[max(0, i - window + 1):i + 1]) for i in range(len(losses))]
plt.figure(figsize=(8, 3.5))
plt.plot(losses, alpha=0.3, label="loss per batch")
plt.plot(smooth, linewidth=2, label=f"average of {window} batches")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Training loss (LoRA)")
plt.legend()
plt.show()

## 5. Les réponses « après »

Posons les **mêmes questions** au modèle fine-tuné, et comparons.

In [ ]:
for q in QUESTIONS:
    after = ask(model, q)
    print(f"Q: {q}")
    print(f"BEFORE: {answers_before[q]}")
    print(f"AFTER : {after}")
    print("-" * 70)

**Que regarder ?**

- La **langue** : le modèle répond-il plus souvent en swahili ?
- Le **style** : les réponses ressemblent-elles aux réponses du jeu de données (listes numérotées, phrases courtes) ?
- Les **erreurs** : un petit modèle et un entraînement de quelques minutes font encore des erreurs de grammaire et de faits.

Le fine-tuning **change le comportement** du modèle (langue, style, format). Il lui apprend peu de **nouvelles connaissances**.

## 6. Sauvegarder et déployer

<p align="center"><img src="https://raw.githubusercontent.com/ganji759/IndabaX-DRC-2026-PyTorch-Workshop/main/images/14_deploiement.png" alt="Du notebook au téléphone : fusion, sauvegarde, ExecuTorch" width="900"></p>

Deux façons de sauvegarder :

1. **L'adaptateur LoRA seul** : seulement `A` et `B`. Petit fichier, facile à partager. Il faut le modèle de base pour l'utiliser.
2. **Le modèle fusionné** : `merge_and_unload()` calcule `W + B·A` dans chaque couche. On obtient un modèle normal, sans LoRA, prêt à déployer.

In [ ]:
import os

def folder_size_mb(path):
    return sum(os.path.getsize(os.path.join(path, f)) for f in os.listdir(path)) / 1e6

# 1) The LoRA adapter only
model.save_pretrained("lora-swahili")
print(f"LoRA adapter : {folder_size_mb('lora-swahili'):7.1f} MB")

# 2) The merged model (W + B @ A), saved in float16 to halve the size
merged = model.merge_and_unload()
merged = merged.to(torch.float16)
merged.save_pretrained("qwen2.5-0.5b-swahili")
tokenizer.save_pretrained("qwen2.5-0.5b-swahili")
print(f"Merged model : {folder_size_mb('qwen2.5-0.5b-swahili'):7.1f} MB")

Vérifions que le modèle sauvegardé fonctionne **seul**, sans LoRA et sans notre code d'entraînement.

In [ ]:
import gc

del model, merged          # free the memory first
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

# Load in float32: small Qwen models can give broken text in pure float16 on a T4
reloaded = AutoModelForCausalLM.from_pretrained("qwen2.5-0.5b-swahili", dtype=torch.float32).to(device)
print(ask(reloaded, QUESTIONS[2]))

### Déployer sur un téléphone avec ExecuTorch

[**ExecuTorch**](https://docs.pytorch.org/executorch/stable/index.html) est la solution de PyTorch pour faire tourner un modèle **sur l'appareil** :
téléphone Android ou iOS, ordinateur, objet connecté. Le modèle tourne **sans Internet**, et les données restent sur l'appareil.
C'est important dans les régions où la connexion est chère ou instable.

Les étapes :

1. **Exporter** le modèle fusionné en un fichier `.pte`, avec quantification (des poids en 4 et 8 bits au lieu de 16 : le fichier devient beaucoup plus petit).
2. **Vérifier** le fichier `.pte` en Python.
3. **Charger** le fichier `.pte` dans une application Android ou iOS.

**Bonus, à faire après l'atelier** (l'installation et l'export prennent plus de 10 minutes).
Voici la commande de la [documentation officielle d'ExecuTorch](https://docs.pytorch.org/executorch/stable/llm/export-llm-optimum.html), adaptée à notre dossier :

```bash
!pip install -q optimum-executorch

!optimum-cli export executorch \
    --model "./qwen2.5-0.5b-swahili" \
    --task "text-generation" \
    --recipe "xnnpack" \
    --use_custom_sdpa \
    --use_custom_kv_cache \
    --qlinear 8da4w \
    --qembedding 8w \
    --output_dir "./qwen_swahili_pte"
```

Puis, pour vérifier le fichier `.pte` en Python (code de la même page) :

```python
from optimum.executorch import ExecuTorchModelForCausalLM

et_model = ExecuTorchModelForCausalLM.from_pretrained("./qwen_swahili_pte")
print(et_model.text_generation(tokenizer=tokenizer, prompt="Habari ya asubuhi", max_seq_len=128))
```

Pour l'application mobile : [Running on Android](https://docs.pytorch.org/executorch/stable/llm/run-on-android.html)
et l'application exemple [LlamaDemo](https://github.com/meta-pytorch/executorch-examples/tree/main/llm/android).

## 7. Résumé

| Étape | Ce qu'il faut retenir | Code |
|---|---|---|
| Modèle pré-entraîné | un `nn.Module` déjà entraîné sur beaucoup de textes | `AutoModelForCausalLM.from_pretrained` |
| Données | conversations → chat template → tokens ; la question est ignorée dans la perte | `apply_chat_template`, `labels = -100` |
| LoRA | on gèle `W`, on apprend `A` et `B` (environ 2 % des paramètres) | `LoraConfig`, `get_peft_model` |
| Entraînement | la même boucle qu'hier, avec la précision mixte | `torch.autocast`, `GradScaler` |
| Sauvegarde | adaptateur seul (petit) ou modèle fusionné (prêt à déployer) | `save_pretrained`, `merge_and_unload` |
| Déploiement | fichier `.pte` sur le téléphone, sans Internet | ExecuTorch |

### Exercices (à la maison)

1. Entraînez sur **5 000 conversations** au lieu de 1 000. Les réponses s'améliorent-elles ?
2. Changez le rang `r` de LoRA (4, puis 64). Combien de paramètres entraînables ? Quelle différence dans les réponses ?
3. Essayez avec des données dans **une autre langue africaine** (par exemple le lingala). Que faut-il changer dans le code ?
4. Faites l'export ExecuTorch (section 6) et testez le fichier `.pte`.

### Pour aller plus loin

- [PyTorch blog : Finetune LLMs on your own consumer hardware](https://pytorch.org/blog/finetune-llms/) (LoRA et QLoRA sur un GPU gratuit)
- [PyTorch : Automatic Mixed Precision](https://docs.pytorch.org/tutorials/recipes/recipes/amp_recipe.html)
- [Hugging Face PEFT : LoRA](https://huggingface.co/docs/peft/main/en/developer_guides/lora)
- [Hugging Face LLM Course, chapitre LoRA](https://huggingface.co/learn/llm-course/chapter11/4)
- [ExecuTorch : Exporting LLMs with Optimum](https://docs.pytorch.org/executorch/stable/llm/export-llm-optimum.html)
- Pour des modèles déjà adaptés aux langues africaines : [AfriqueLLM (McGill-NLP)](https://huggingface.co/McGill-NLP) et [Masakhane](https://www.masakhane.io/).

---

**Crédits.** Modèle : [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) (Qwen, licence Apache-2.0).
Données : [swahili-instruction-22k](https://huggingface.co/datasets/regnant-io/swahili-instruction-22k) (regnant-io, licence Apache-2.0).
Bibliothèques : PyTorch, Hugging Face `transformers`, `datasets` et `peft`. Schémas faits avec [Excalidraw](https://excalidraw.com).

**Merci d'avoir participé à l'atelier PyTorch de l'IndabaX RDC 2026 !** 🚀